# LibraryGuard — Pipeline Walkthrough

This notebook demonstrates the core components end-to-end on **synthetic data**,
so it runs anywhere without downloading datasets:

1. configuration
2. tracker behavior (anonymous IDs)
3. behavior evidence scoring
4. temporal event aggregation
5. feature extraction for the trainable temporal model
6. evaluation metrics

For full video processing see `scripts/predict.py`; for training see
`scripts/train.py` and `docs/datasets.md`.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

from src.utils.config import load_config

config = load_config()
print("Tracker:", config.tracking.tracker)
print("Drinking min duration:", config.behavior.drinking.min_duration_seconds, "s")
print("Sleeping min duration:", config.behavior.sleeping.min_duration_seconds, "s")
print("Phone min duration:  ", config.behavior.phone_usage.min_duration_seconds, "s")

## 2. Anonymous multi-person tracking

In [ ]:
import numpy as np
from src.tracking.tracker import create_tracker
from src.utils.constants import Box

tracker = create_tracker("bytetrack", min_hits=2, max_age=5)

def b(x1, y1, x2, y2):
    return Box(x1, y1, x2, y2)

# Two people: one still, one walking slowly across the frame.
for i in range(12):
    walker = b(0.1 + 0.03 * i, 0.2, 0.25 + 0.03 * i, 0.8)
    still  = b(0.65, 0.25, 0.80, 0.85)
    active = tracker.update([walker, still], [0.92, 0.88])
    if i == 11:
        for t in active:
            c = t.bbox.center
            print(f"Person #{t.track_id}: center=({c.x:.2f}, {c.y:.2f})  movement={t.movement_over_window(5):.4f}")

## 3. Behavior evidence scoring

Each detector combines **multiple cues**. Note how a bottle on a table scores
near zero while a cup at the mouth scores high.

In [ ]:
from src.behavior.drinking import DrinkingDetector
from tests.conftest import make_box, make_observation, make_pose, object_tuple

detector = DrinkingDetector()

table = make_observation(
    objects=[object_tuple(make_box(0.60, 0.75, 0.68, 0.90), "bottle", 0.9)],
    pose=make_pose(),
)
drinking = make_observation(
    objects=[object_tuple(make_box(0.21, 0.14, 0.29, 0.22), "cup", 0.95)],
    pose=make_pose(nose=(0.25, 0.18), left_wrist=(0.26, 0.20), right_wrist=(0.35, 0.40)),
)

s1, s2 = detector.score(table), detector.score(drinking)
print(f"bottle on table : {s1.confidence:.2f}  {s1.components}")
print(f"cup at the mouth: {s2.confidence:.2f}  {s2.components}")

## 4. Temporal event aggregation

The engine turns per-frame scores into events using smoothing, minimum
duration and cooldowns. A short blip is discarded; a sustained behavior
becomes one event.

In [ ]:
from src.behavior.base import BehaviorScore
from src.behavior.engine import BehaviorEngine, PerBehaviorConfig
from src.behavior.drinking import DrinkingDetector

class BlipThenSustain(DrinkingDetector):
    def score(self, observation):
        raw = 0.95 if observation.frame_index >= 15 else (0.9 if observation.frame_index % 10 < 2 else 0.05)
        return BehaviorScore(self.key, raw)

engine = BehaviorEngine(
    detectors=[BlipThenSustain()],
    configs={"drinking": PerBehaviorConfig(min_duration_seconds=0.8, confidence_threshold=0.6,
                                           cooldown_seconds=5.0, min_consecutive_frames=3)},
    fps=10.0,
    min_event_duration_seconds=0.0,
)

for i in range(40):
    obs = make_observation(frame_index=i, timestamp=i / 10.0)
    engine.update(i, i / 10.0, [obs])

for event in engine.finalize():
    print(event.summary)

## 5. Feature extraction for the trainable temporal model

The same evidence the rule engine uses becomes input features for the GRU
classifier — one shared, honest representation.

In [ ]:
from src.models.temporal import FEATURE_NAMES, FeatureExtractor

fe = FeatureExtractor()
features = fe.extract(drinking)
for name, value in zip(FEATURE_NAMES, features):
    print(f"{name:>22}: {value:+.3f}")

## 6. Evaluation metrics

Metrics are computed from supplied predictions/labels only — never invented.

In [ ]:
from src.evaluation.metrics import compute_classification_metrics, compute_event_metrics

report = compute_classification_metrics(
    y_true=[0, 0, 1, 1, 2, 2, 0, 2],
    y_pred=[0, 1, 1, 1, 2, 0, 0, 2],
    labels=["drinking", "sleeping", "phone_usage"],
)
print(f"accuracy={report.accuracy:.2f}  macro-F1={report.macro_f1:.2f}")
for label in report.labels:
    print(f"  {label:>12}: P={report.precision[label]:.2f} R={report.recall[label]:.2f} F1={report.f1[label]:.2f}")

ev = compute_event_metrics(
    pred_events=[{"event": "drinking", "time_start": 10.2, "time_end": 13.0, "confidence": 0.9}],
    gt_events=[{"event": "drinking", "time_start": 10.0, "time_end": 13.5}],
)
print("event-level:", ev.to_dict()["per_event_type"]["drinking"])

---

**Next steps:** `python scripts/predict.py --source your_video.mp4` for full
videos, `python scripts/train.py ...` to train the temporal classifier, and
`streamlit run app/streamlit_app.py` for the dashboard.